# Non-CSS surface-code decoding

This notebook decodes the full, non-CSS GKP surface-code lattice. It mirrors the CSS-sector example, but keeps the coupled `q` and `p` quadratures together.

In [ ]:
using Random
using LinearAlgebra
using LatticeDecoder

Random.seed!(2);

Build a distance-3 surface code and derive the full parity-check matrix `H` and correction generator `G` in the `qqpp` convention.

In [ ]:
d = 3

code = GKP_Surface_Code(d, false);
M = code.code;
J = code.J;

H = -M * J;
G = J * inv(M);
logical_check = inv(H);

(check_matrix_size = size(H), generator_size = size(G))

Draw one full displacement vector and decode it with serial belief propagation.

In [ ]:
noise_std = 0.05
max_iter = size(H, 2)
decoder = "lsd"
search_interval = 1.0

error_vector = sample_error(noise_std, size(H, 2));
received = copy(error_vector);

tanner_graph = initialize_tanner_graph(H);
bp_estimate = run_serial_belief_propagation!(
    tanner_graph,
    received,
    noise_std,
    max_iter,
    decoder;
    search_interval = search_interval,
);

decoded_integer_correction = hard_decision(bp_estimate, H)

Check whether the final residual is logically trivial for the full lattice.

In [ ]:
function is_not_logical_error(logical_check, residual; atol = 1e-5)
    logical_coordinates = logical_check' * residual
    return all(abs(x - round(x)) < atol for x in logical_coordinates)
end

correction = received - G * decoded_integer_correction;
residual = error_vector - correction;
logical_coordinates = logical_check' * residual;

summary = (
    distance = d,
    noise_std = noise_std,
    symbol_errors = count_symbol_errors(decoded_integer_correction),
    logical_success = is_not_logical_error(logical_check, residual),
    max_logical_residual = maximum(abs.(logical_coordinates .- round.(logical_coordinates))),
)